# PRIMA PROVA RANDOM FOREST

---

Leggo il nostro dict in `.pkl`

In [1]:
import pickle as pkl

with open('data/ottani_NIR.pkl', 'rb') as f:
    dizionario = pkl.load(f)
    
# dati train
train_ones = dizionario['train']['NIR']['ones']
train_zeros = dizionario['train']['NIR']['zeros']
train_labels = dizionario['train']['labels']

# dati test
test_ones = dizionario['test']['NIR']['ones']
test_zeros = dizionario['test']['NIR']['zeros']
test_labels = dizionario['test']['labels']


In [2]:
import numpy as np

train_x = np.concatenate((train_zeros, train_ones))
test_x = np.concatenate((test_zeros,test_ones))


---

## SMOOTHING 

provo a filtrare i dati con Savitzky-Golay

In [3]:
from scipy.signal import savgol_filter

window_size = 11
poly_order = 3

for i in range(train_x.shape[0]):
    train_x[i] = savgol_filter(train_x[i], window_size, poly_order)

for j in range(test_x.shape[0]):
    test_x[j] = savgol_filter(test_x[j], window_size, poly_order)

---

In [4]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.ensemble import RandomForestClassifier as RFC

76 minuti

In [5]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=10, random_state=42)

# PCA
N_COMPONENTS_OPTIONS = [2, 5, 7, 19, 37, None]
# ESTIMATOR
N_ESTIMATOR_OPTIONS = [50, 100, 150, 250, 300, 500]
OBJ_FUNCTION_OPTIONS = ['gini', 'entropy']
MAX_FEATURES_OPTIONS = ["sqrt", "log2"]
MAX_DEPTH_OPTIONS = [2, 3, 5]
MIN_SAMPLES_LEAF_OPTIONS = [1, 2, 5]

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", MinMaxScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=42)),
    
    # Step 3: Classificatore BOOTSTRAP SEMPRE TRUE PERCHÉ HO POCHI SAMPLES (44)
    ("classify", RFC(random_state=42, bootstrap=True, 
                     criterion='entropy', 
                     max_depth=3,
                     max_features='sqrt',
                     min_samples_leaf=1,
    )) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{   
    # Per provare con diversi numeri di componenti (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per provare parametri del classificatore
    "classify__n_estimators": N_ESTIMATOR_OPTIONS,
},
{   
    # Per saltare la PCA
    "reduce_dim": ['passthrough'], 
    
    # Per provare parametri del classificatore
    "classify__n_estimators": N_ESTIMATOR_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(test_x, test_labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale:.4f}")

La miglior configurazione: {'classify__n_estimators': 250, 'reduce_dim__n_components': 19}
Fornisce accuracy in validation: 0.9250
Risultato sul set indipendente (Segnale A): 0.8333


In [6]:
import pandas as pd
# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)
results_df.to_pickle("results/results-rf-savgol-fine-GridSearch.pkl")

In [7]:
import pandas as pd

results_df = pd.DataFrame(grid.cv_results_)
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_reduce_dim__n_components', 
    'param_classify__n_estimators', 
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'std_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')
renamed = analysis.rename(columns={
    'param_scaling': 'scaler',
    'param_reduce_dim__n_components': 'pca_n_components', 
    'param_classify__n_estimators': 'n_estimators', 
    'mean_test_score': 'mean_score', 
    'std_test_score': 'std_score', 
    'mean_test_sensitivity': 'mean_sensitivity',
    'std_test_sensitivity': 'std_sensitivity',
    'rank_test_score': 'rank_score'
})

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
renamed.head(20)

Numero totale di configurazioni provate: 42


,pca_n_components,n_estimators,mean_score,std_score,mean_sensitivity,std_sensitivity,rank_score
21,19,250,0.925000,0.076376,0.966667,0.084984,1
27,19,300,0.922917,0.085670,0.962500,0.100260,2
38,NaN,150,0.920833,0.088290,0.950000,0.100000,3
33,19,500,0.918750,0.093611,0.954167,0.124931,4
39,NaN,250,0.916667,0.090331,0.941667,0.105738,5
37,NaN,100,0.914583,0.095448,0.950000,0.100000,6
36,NaN,50,0.914583,0.092679,0.945833,0.102993,6
40,NaN,300,0.912500,0.089268,0.933333,0.110554,8
15,19,150,0.912500,0.089268,0.958333,0.093169,8
41,NaN,500,0.910417,0.097071,0.929167,0.121550,10
